# Initial Analysis — Reddit Influencer Comments

**Purpose.** Before any NLP modelling, this notebook does a sanity-and-design audit on the scraped corpus. We want to know:

1. Is the data *clean enough* to feed into a zero-shot classifier? (deduplication, deleted comments, false positives, length outliers)
2. Is the **2×2 design** (Influencer Tier × Subreddit Stratum) statistically viable? Each cell needs enough comments to estimate a mean — rule of thumb ≥ 300 per cell, ≥ 1,000 is comfortable.
3. Are there **temporal anomalies** (e.g. a single drama event spiking one influencer's comment volume) that we'll need to control for?
4. Does the empirical data force us to **adjust the research question**, and if so, what are the options?

The notebook ends with a decision-summary cell that translates findings into research-design implications.


## 1. Setup & data load

We use only standard libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`). No NLP yet — that's the next notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 200)
sns.set_style("whitegrid")

# Path to the scraped CSV. Adjust if your folder layout is different.
RAW_PATH = "comments_raw_full.csv"

raw = pd.read_csv(RAW_PATH)
print(f"Raw rows (with query-variant duplicates): {len(raw):,}")
print(f"Unique comment IDs:                       {raw['id'].nunique():,}")
print(f"Columns:                                  {list(raw.columns)}")


## 2. Deduplicate on comment ID

The scraper runs every search-query variant separately (e.g. it queries both `"Matilda Djerf"` and `"Djerf"`), so the same comment can appear multiple times in the raw file — once per variant that matched. We dedupe on `id` (Reddit's globally unique comment identifier) before any downstream analysis.

We keep the first occurrence's `search_query` value just so we can audit which query variant produced each retained row, but for analysis we don't care which variant matched.


In [ ]:
df = raw.drop_duplicates(subset="id").copy().reset_index(drop=True)
print(f"After dedup: {len(df):,} unique comments")
print(f"Duplicates removed by query-variant overlap: {len(raw) - len(df):,}")


## 3. Cleaning pipeline

Standard text-mining hygiene. Each filter removes a known noise category:

- **Empty / `[deleted]` / `[removed]`** — Reddit shows these strings instead of the original body when a user deletes a comment or a moderator removes one. They contain no analyzable text.
- **AutoModerator and bot replies** — boilerplate, not user expression.
- **Length floor (≥ 3 tokens)** — single-word replies like "lol", "this", or just a name are too thin to classify reliably with zero-shot.
- **Length ceiling (≤ 1000 tokens)** — extremely long comments are usually pasted articles or quoted block-text, not authentic user reactions.
- **Risky-query false-positive re-check** — for single-word queries (e.g. just "Djerf") we verify the canonical name token actually appears in the body. Currently we treat these case-by-case.


In [ ]:
# Counters for the report
n0 = len(df)

# Drop deleted/removed/empty
df = df[~df["body"].fillna("").isin(["", "[deleted]", "[removed]"])].copy()
n1 = len(df)

# Drop AutoModerator
df = df[df["author"] != "AutoModerator"].copy()
n2 = len(df)

# Token length
df["body_len"] = df["body"].str.split().str.len()
df = df[(df["body_len"] >= 3) & (df["body_len"] <= 1000)].copy()
n3 = len(df)

# Encode IV and control as 0/1 — useful later for regression
df["tier_mega"] = (df["influencer_tier"] == "mega").astype(int)
df["sub_snark"] = (df["subreddit_stratum"] == "snark").astype(int)

print(f"Start                : {n0:,}")
print(f"After deleted/removed: {n1:,}  (-{n0-n1:,})")
print(f"After AutoMod        : {n2:,}  (-{n1-n2:,})")
print(f"After length filter  : {n3:,}  (-{n2-n3:,})")
print(f"\nFinal cleaned corpus: {len(df):,} comments")


## 4. The headline diagnostic — the 2×2

Your hypotheses (H1a, H1b) compare Mega vs. Micro tier on envy outcomes, with subreddit stratum (Discussion vs. Snark) as a control. The mixed-effects model needs each of the four cells to have enough comments to estimate a mean reliably.

**Rule of thumb:**
- ≥ 1,000 per cell → comfortable, full statistical power
- 300 – 1,000 per cell → workable, wider confidence intervals
- < 300 per cell → underpowered, results will be unstable
- < 50 per cell → cannot meaningfully estimate; effectively missing


In [ ]:
cell_counts = (df.groupby(["influencer_tier", "subreddit_stratum"])
                 .size()
                 .unstack(fill_value=0))
print("Cell counts (Tier × Stratum):")
print(cell_counts)

# Heatmap so the imbalance is unmissable
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(cell_counts, annot=True, fmt=",d", cmap="YlOrRd",
            cbar_kws={"label": "n comments"}, ax=ax)
ax.set_title("Comments per design cell\n(rule of thumb: ≥ 300 needed; ≥ 1,000 comfortable)")
ax.set_xlabel("Subreddit stratum (control)")
ax.set_ylabel("Influencer tier (IV)")
plt.tight_layout()
plt.show()


**Read the table carefully.** Three of the four cells are healthy. The Micro × Snark cell is essentially empty — *snark subreddits don't discuss Micro/Meso-tier influencers*, because the genre needs a famous-enough target to mock. This is a real social-media phenomenon, not a scraper bug, but it has consequences for the analysis. We'll come back to this in the decision summary.


## 5. Per-influencer volumes

How balanced is the corpus *within* each tier? If a single influencer dominates a tier (e.g. Alix Earle = 80% of all Mega comments), then any "Mega-tier effect" we find is really an Alix-Earle effect. We want to see whether the tier signal is broad-based.


In [ ]:
inf_counts = (df.groupby(["influencer_tier", "matched_influencer"])
                .size()
                .reset_index(name="n")
                .sort_values(["influencer_tier", "n"], ascending=[True, False]))

fig, ax = plt.subplots(figsize=(10, 7))
colors = inf_counts["influencer_tier"].map({"mega": "#1f77b4", "micro": "#ff7f0e"})
ax.barh(inf_counts["matched_influencer"], inf_counts["n"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("n comments")
ax.set_title("Comments per influencer (blue = Mega, orange = Micro)")
# Reference lines for sample-size rules of thumb
for x, label in [(50, "n=50"), (300, "n=300"), (1000, "n=1,000")]:
    ax.axvline(x, color="grey", linestyle="--", alpha=0.5)
    ax.text(x, len(inf_counts)-0.5, label, fontsize=8, color="grey")
plt.tight_layout()
plt.show()

print("\nInfluencers below 50 comments (consider dropping from analysis):")
print(inf_counts[inf_counts["n"] < 50].to_string(index=False))


## 5.1 Kackie Reviews Beauty — comment impact assessment

**Context.** Kackie Reviews Beauty's Instagram account was terminated during the study period, which means the follower-count classification used to assign the influencer tier (mega ≥ 1M vs micro/meso 50K–300K) can no longer be independently verified from the account itself. This section counts the affected comments so we can decide whether to (a) retain them and document the caveat in the Limitations chapter, or (b) drop them from the analytical corpus. The decision hinges on how large a share of the corpus these comments represent and whether removing them would materially change the tier balance.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5.1 Kackie Reviews Beauty — impact assessment
# ─────────────────────────────────────────────────────────────────

# The exact string used in the matched_influencer column. Adjust
# if the scraper stored a different capitalisation or spacing.
KACKIE_NAME = 'Kackie Reviews Beauty'

# Fuzzy fallback in case the name is stored slightly differently
candidates = [n for n in df['matched_influencer'].dropna().unique()
              if 'kackie' in n.lower()]
if KACKIE_NAME not in df['matched_influencer'].unique() and candidates:
    print(f'Note: exact match on {KACKIE_NAME!r} not found. '
          f'Fuzzy candidates: {candidates}')
    KACKIE_NAME = candidates[0]
    print(f'Using {KACKIE_NAME!r} for the assessment.')

kackie   = df[df['matched_influencer'] == KACKIE_NAME].copy()
n_kackie = len(kackie)
n_total  = len(df)
pct      = 100 * n_kackie / n_total if n_total else 0

print('=' * 66)
print('Kackie Reviews Beauty — comment impact assessment')
print('=' * 66)
print(f'Total corpus size:            {n_total:>6,} comments')
print(f'Comments matched to Kackie:   {n_kackie:>6,} comments')
print(f'Share of total corpus:        {pct:>6.2f}%')
print()

if n_kackie == 0:
    print('No comments matched to Kackie in the current corpus — '
          'no action needed.')
else:
    # Current tier assignment
    tiers = list(kackie['influencer_tier'].dropna().unique())
    print(f'Currently classified as tier: {tiers}')
    print()

    # Breakdown by subreddit
    print('Comments per subreddit:')
    for sub, n in (kackie.groupby('subreddit').size()
                         .sort_values(ascending=False).items()):
        print(f'  {sub:<28s} {n:>4d}')
    print()

    # Breakdown by subreddit stratum
    if 'subreddit_stratum' in kackie.columns:
        print('Comments per stratum:')
        for stratum, n in (kackie.groupby('subreddit_stratum').size()
                                  .items()):
            print(f'  {stratum:<28s} {n:>4d}')
        print()

    # Time distribution
    if 'month_window' in kackie.columns:
        print('Comments per month_window:')
        for mw, n in (kackie.groupby('month_window').size()
                            .sort_index().items()):
            print(f'  {mw:<15s} {n:>4d}')
        print()

    # Tier-balance impact of a hypothetical drop
    df_without = df[df['matched_influencer'] != KACKIE_NAME]
    before = df.groupby('influencer_tier').size()
    after  = df_without.groupby('influencer_tier').size()
    print('Tier balance BEFORE vs AFTER dropping Kackie:')
    for t in sorted(set(list(before.index) + list(after.index))):
        b = int(before.get(t, 0))
        a = int(after.get(t, 0))
        change = b - a
        print(f'  {t:<8s} before: {b:>5,d}   after: {a:>5,d}   '
              f'change: -{change:>3,d}')
    print()

    # Interpretive guidance based on the corpus share
    print('-' * 66)
    if pct < 2:
        print(f'Interpretation: Kackie is {pct:.2f}% of the corpus '
              '(< 2%).')
        print('Dropping would have minimal effect on sample size or tier')
        print('balance. Retention with a documented caveat in the')
        print('Limitations chapter is equally defensible.')
    elif pct < 5:
        print(f'Interpretation: Kackie is {pct:.2f}% of the corpus '
              '(2–5%).')
        print('Dropping is a reasonable option that leaves sample size')
        print('largely intact; retention with a caveat is also defensible.')
    else:
        print(f'Interpretation: Kackie is {pct:.2f}% of the corpus '
              '(>= 5%).')
        print('This is substantial. Retention with a documented caveat')
        print('is preferable to preserve statistical power, unless the')
        print('tier assignment itself is dubious.')

### Decision guidance

Two options based on the numbers printed above:

1. **Retain with caveat.** Add a paragraph to the Methods chapter describing the account termination, and note in the Limitations chapter that Kackie's tier assignment reflects the account's documented pre-termination follower count rather than a current verification. This is the standard approach when a small number of accounts become inaccessible mid-project. It is preferable when the corpus share is small (< 5%) or when there is no reason to doubt the pre-termination tier classification.

2. **Drop and re-run.** If the tier assignment is genuinely uncertain (e.g., you do not know whether Kackie was mega or micro before termination), remove these comments from the analytical corpus and re-run Notebooks 02c, 02d, 03, and 04. The impact on results depends on the corpus share reported above; based on the effect sizes reported in the v4 progress report, a drop of < 5% is unlikely to shift H1/H3 coefficients or the LCA typology substantively, but the reported sample counts and χ² values will change slightly.

The code cell below performs the drop if you decide to remove Kackie's comments. It is commented out by default; uncomment the two active lines and re-run this notebook from §9 (save the cleaned corpus) onward. Note that re-running Notebook 02c requires re-scoring the reduced corpus through the LLM API.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# OPTIONAL: drop Kackie's comments from the analytical corpus.
#
# Uncomment the two lines below to apply the drop, then re-run the
# save step in §9 to write the reduced corpus to comments_clean.csv.
# You will also need to re-run Notebooks 02c, 02d, 03, and 04 with
# the reduced corpus if you choose this route.
# ─────────────────────────────────────────────────────────────────

# df = df[df['matched_influencer'] != KACKIE_NAME].copy().reset_index(drop=True)
# print(f'Dropped Kackie\'s comments. New corpus size: {len(df):,} comments')

## 6. Per-subreddit volumes

Same balance question for the venue side. We've stratified subreddits into Discussion (control = 0) and Snark (control = 1). Are both strata broadly populated, or is one stratum carried by a single subreddit?


In [ ]:
sub_counts = (df.groupby(["subreddit_stratum", "subreddit"])
                .size()
                .reset_index(name="n")
                .sort_values(["subreddit_stratum", "n"], ascending=[True, False]))

fig, ax = plt.subplots(figsize=(10, 7))
colors = sub_counts["subreddit_stratum"].map({"discussion": "#2ca02c", "snark": "#d62728"})
ax.barh(sub_counts["subreddit"], sub_counts["n"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("n comments")
ax.set_title("Comments per subreddit (green = Discussion, red = Snark)")
for x in [50, 300, 1000]:
    ax.axvline(x, color="grey", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("\nSubreddit-level cumulative share of corpus:")
sub_counts["cum_pct"] = (sub_counts["n"].cumsum() / sub_counts["n"].sum() * 100).round(1)
print(sub_counts.to_string(index=False))


## 7. Time-series check — drama spikes

A pet concern in influencer-marketing research: a single viral drama event (a public feud, a scandal, a brand collapse) can produce a one-month spike that overwhelms 30 months of baseline discussion. If your "Mega-tier effect" comes mostly from one Jaclyn Hill drama week in March 2024, that's not a stable theoretical signal — it's a news-cycle artefact.

We plot monthly counts faceted by tier so we can eyeball this. Spiky lines = drama-driven; flat-ish lines = stable baseline discussion.


In [ ]:
ts = (df.groupby(["month_window", "influencer_tier"])
        .size()
        .reset_index(name="n"))
ts["month_dt"] = pd.to_datetime(ts["month_window"])

fig, ax = plt.subplots(figsize=(11, 4))
for tier, sub in ts.groupby("influencer_tier"):
    ax.plot(sub["month_dt"], sub["n"], marker="o", label=tier)
ax.set_title("Monthly comment volume by Influencer Tier")
ax.set_ylabel("n comments")
ax.legend(title="Tier")
plt.tight_layout()
plt.show()

# Also faceted per top-5 influencer to identify which names drive the spikes
top5 = (df.groupby("matched_influencer").size()
          .sort_values(ascending=False).head(5).index.tolist())
ts2 = (df[df["matched_influencer"].isin(top5)]
       .groupby(["month_window", "matched_influencer"])
       .size().reset_index(name="n"))
ts2["month_dt"] = pd.to_datetime(ts2["month_window"])

fig, ax = plt.subplots(figsize=(11, 4))
for inf, sub in ts2.groupby("matched_influencer"):
    ax.plot(sub["month_dt"], sub["n"], marker="o", label=inf, alpha=0.85)
ax.set_title("Monthly volume — top 5 influencers")
ax.set_ylabel("n comments")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 8. Body-length sanity & qualitative spot check

A 200-character snippet from each design cell, to confirm what the classifier will actually be reading. Eyeball this — does it look like authentic user discussion?


In [ ]:
# Length distribution
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(df["body_len"], bins=60)
ax.set_yscale("log")
ax.set_xlabel("comment length (tokens)")
ax.set_ylabel("count (log scale)")
ax.set_title("Comment length distribution")
plt.tight_layout()
plt.show()

print(df["body_len"].describe().round(1))

print("\n=== Random samples per design cell ===")
for (tier, stratum), sub in df.groupby(["influencer_tier", "subreddit_stratum"]):
    take = min(3, len(sub))
    if take == 0:
        continue
    print(f"\n--- {tier.upper()} × {stratum.upper()} (n={len(sub):,}) ---")
    for _, r in sub.sample(take, random_state=42).iterrows():
        body = str(r["body"])[:200].replace("\n", " ")
        print(f"  [{r['matched_influencer']} | r/{r['subreddit']}] {body}")


## 9. Save the cleaned corpus

We persist the cleaned dataframe as `comments_clean.csv` so the next notebook (NLP classification) can pick up from here without re-running this pipeline.


In [ ]:
out_cols = ["id", "subreddit", "subreddit_stratum", "sub_snark",
            "matched_influencer", "influencer_tier", "tier_mega",
            "author", "body", "body_len", "score",
            "created_utc", "created_iso", "month_window", "permalink"]
df[out_cols].to_csv("comments_clean.csv", index=False)
print(f"Saved {len(df):,} cleaned comments → comments_clean.csv")


## 10. Decision summary — does the research question require adjustment?

### What the data confirms

- Pipeline executes end-to-end. ~8,000 cleaned comments, no garbage rows, body content is authentic.
- All 15 subreddits returned data. 19 of 20 influencers returned data (Samantha March returned essentially nothing and is dropped from the analytical list).
- Three of the four design cells (Mega × Discussion, Mega × Snark, Micro × Discussion) are statistically healthy with > 2,000 comments each.
- Within both Mega and Micro tiers, at least 4–5 influencers carry substantial volume, mitigating any single-influencer artefact in the tier effect.

### What the data forces the analysis to confront

The Micro × Snark cell contains ~20 comments. At this n, the cell mean cannot be estimated meaningfully, the Tier × Stratum interaction cannot be tested, and any "control for venue" coefficient on the Micro side is unstable. This is not a scraper failure — it is a structural feature of Reddit: snark subreddits exist to mock public figures, and Micro/Meso-tier influencers are not famous enough to draw sustained snark attention.

### Three options for adjusting the research question

**Option 1 — Drop the snark stratum from the main hypothesis test.**
Run H1a and H1b on Discussion subreddits only (Mega ≈ 2,079 vs Micro ≈ 2,511 — well-balanced and high-powered). Report Snark separately as a *Mega-only descriptive supplement* showing how venue tone differs *within* the Mega tier. This is the most statistically defensible move and aligns the analysis with where the data lie. The downside is the loss of the Stratum control variable in the main test; this is mitigated by including subreddit-level random intercepts in the mixed-effects model.

**Option 2 — Reframe the IV as Tier–Venue Affinity rather than Tier alone.**
The data suggest that Mega influencers are discussed in *both* discussion and snark venues, while Micro influencers are almost exclusively in discussion venues. This asymmetry itself can be modelled as a construct.

**Option 3 — Extend data collection to increase the Micro × Snark cell size.**
Not pursued in this dissertation due to time and API-quota constraints, but noted as a future-research direction.

**Chosen option: Option 1** (see Notebook 02c §2 where the snark drop is executed).
